# Prompt 9: Built-in Spark SQL Functions Reference
## Databricks Certified Associate Developer for Apache Spark
### Topic 2 — Using Spark SQL (20%)

---

**Import pattern — memorise this for the exam:**
```python
from pyspark.sql.functions import (
    col, lit, expr,
    # String
    upper, lower, trim, ltrim, rtrim, length, initcap,
    substring, split, concat, concat_ws,
    regexp_replace, regexp_extract, translate, lpad, rpad,
    # Numeric
    round, bround, abs, ceil, floor, sqrt, pow, log, mod,
    # Date/time
    current_date, current_timestamp, to_date, to_timestamp,
    date_format, date_add, date_sub, datediff, months_between,
    year, month, dayofmonth, dayofweek, hour, minute, second,
    unix_timestamp, from_unixtime,
    # Conditional / null
    when, coalesce, nullif, isnull, isnotnull,
    # Aggregate
    count, countDistinct, sum, avg, mean, min, max,
    collect_list, collect_set, first, last, stddev, variance,
    # Array / map
    array, explode, explode_outer, posexplode,
    array_contains, size, flatten, sort_array,
    array_distinct, array_union, map_keys, map_values,
)
```

**Key exam fact:** Every function in `pyspark.sql.functions` is also available as a SQL string
inside `spark.sql()`, `selectExpr()`, or `expr()` — you do **not** need to import them for SQL usage.

```python
# Python API                        # SQL string equivalent
upper(col('name'))               # 'UPPER(name)'
round(col('salary'), 2)          # 'ROUND(salary, 2)'
date_add(col('dt'), 7)           # 'DATE_ADD(dt, 7)'
when(col('x') > 0, 'pos')        # 'CASE WHEN x > 0 THEN "pos" END'
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName('BuiltinFunctions') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Sample DataFrames used throughout
people = spark.createDataFrame([
    (1, '  Alice Smith  ', 'alice@example.com',  32, 95000.567),
    (2, 'BOB JONES',       'bob@test.org',        28, 72000.123),
    (3, 'carol white',     None,                  45, 110000.0),
    (4, 'Dave  Brown',     'dave@co.uk',          None, None),
], ['id', 'name', 'email', 'age', 'salary'])

events = spark.createDataFrame([
    (1, '2024-01-15', '2024-01-15 08:30:00', 1705305000),
    (2, '2024-03-20', '2024-03-20 14:45:30', 1710944730),
    (3, '2023-11-05', '2023-11-05 23:59:59', 1699228799),
], ['event_id', 'event_date_str', 'event_ts_str', 'epoch'])

items = spark.createDataFrame([
    (1, ['apple', 'banana', 'cherry'], {'a': 1, 'b': 2}),
    (2, ['apple', 'date'],             {'c': 3}),
    (3, [],                            {}),
], ['id', 'fruits', 'scores'])

print('people schema:'); people.printSchema()
people.show()
print('events schema:'); events.printSchema()
events.show()
print('items schema:'); items.printSchema()
items.show()

## 1. String Functions

| Function | Signature | Description |
|----------|-----------|-------------|
| `upper` | `upper(col)` | Converts string to uppercase |
| `lower` | `lower(col)` | Converts string to lowercase |
| `trim` | `trim(col)` | Removes leading and trailing spaces |
| `ltrim` | `ltrim(col)` | Removes leading spaces only |
| `rtrim` | `rtrim(col)` | Removes trailing spaces only |
| `length` | `length(col)` | Number of characters in string |
| `initcap` | `initcap(col)` | Capitalises first letter of each word |
| `substring` | `substring(col, pos, len)` | Extract substring (1-based index) |
| `split` | `split(col, pattern)` | Split by regex pattern → ArrayType |
| `concat` | `concat(col1, col2, ...)` | Concatenate strings (NULL propagates) |
| `concat_ws` | `concat_ws(sep, col1, ...)` | Concatenate with separator (NULLs skipped) |
| `regexp_replace` | `regexp_replace(col, pattern, replacement)` | Replace matches with replacement string |
| `regexp_extract` | `regexp_extract(col, pattern, group_idx)` | Extract capture group from regex match |
| `translate` | `translate(col, matching, replace)` | Character-by-character substitution |
| `lpad` | `lpad(col, len, pad)` | Left-pad string to length |
| `rpad` | `rpad(col, len, pad)` | Right-pad string to length |

In [ ]:
# Cell 2: String functions demo
from pyspark.sql.functions import (
    upper, lower, trim, ltrim, rtrim, length, initcap,
    substring, split, concat, concat_ws,
    regexp_replace, regexp_extract, translate, lpad, rpad
)

# Basic case/trim/length
print('=== upper / lower / initcap / trim / length ===')
people.select(
    col('name'),
    upper(col('name')).alias('upper_name'),
    lower(col('name')).alias('lower_name'),
    initcap(trim(col('name'))).alias('initcap_trimmed'),
    length(trim(col('name'))).alias('name_len')
).show(truncate=False)

# substring / split / concat / concat_ws
print('=== substring / split / concat / concat_ws ===')
people.select(
    col('name'),
    substring(col('name'), 1, 5).alias('first5'),
    split(trim(col('name')), ' ').alias('name_parts'),
    concat(col('id').cast('string'), F.lit(': '), trim(col('name'))).alias('id_name'),
    concat_ws(' | ', col('id').cast('string'), trim(col('name')), col('email')).alias('all_fields')
).show(truncate=False)

# regexp_replace / regexp_extract
print('=== regexp_replace / regexp_extract ===')
people.filter(col('email').isNotNull()).select(
    col('email'),
    regexp_replace(col('email'), r'@.*', '').alias('username'),
    regexp_extract(col('email'), r'@([^.]+)\.', 1).alias('domain_name')
).show()

# lpad / rpad / translate
print('=== lpad / rpad / translate ===')
people.select(
    col('id'),
    lpad(col('id').cast('string'), 5, '0').alias('padded_id'),
    rpad(trim(col('name')), 20, '.').alias('padded_name'),
    translate(lower(trim(col('name'))), 'aeiou', '*****').alias('vowels_replaced')
).show(truncate=False)

## 2. Numeric / Math Functions

| Function | Signature | Description |
|----------|-----------|-------------|
| `round` | `round(col, scale)` | Round to scale decimal places (half-up) |
| `bround` | `bround(col, scale)` | Banker's rounding (half-to-even) |
| `abs` | `abs(col)` | Absolute value |
| `ceil` | `ceil(col)` | Ceiling (round up to nearest integer) |
| `floor` | `floor(col)` | Floor (round down to nearest integer) |
| `sqrt` | `sqrt(col)` | Square root |
| `pow` | `pow(col, exp)` | Raise to power |
| `log` | `log(base, col)` | Logarithm with given base; `log(col)` = natural log |
| `mod` | `col % n` or `mod(col, n)` | Modulo (remainder) |

> **Exam note:** `round()` uses HALF_UP rounding; `bround()` uses HALF_EVEN (banker's rounding).
> `round(2.5, 0)` → 3.0; `bround(2.5, 0)` → 2.0

In [ ]:
# Cell 3: Numeric functions demo
from pyspark.sql.functions import round, bround, abs as F_abs, ceil, floor, sqrt, pow as F_pow, log, mod

nums = spark.createDataFrame([
    (1, 2.5,  -7.3,  100.0),
    (2, 3.5,   0.0,   81.0),
    (3, 2.449, 15.7,  64.0),
    (4, -4.6,  -0.1,  49.0),
], ['id', 'x', 'y', 'z'])

nums.select(
    col('x'),
    round(col('x'), 0).alias('round_x'),      # 2.5 → 3.0, 3.5 → 4.0
    bround(col('x'), 0).alias('bround_x'),     # 2.5 → 2.0, 3.5 → 4.0 (banker's)
    ceil(col('x')).alias('ceil_x'),
    floor(col('x')).alias('floor_x'),
    F_abs(col('y')).alias('abs_y'),
    sqrt(col('z')).alias('sqrt_z'),
    F_pow(col('z'), 0.5).alias('pow_half'),    # same as sqrt
    (col('id') % 2).alias('id_mod2'),
    log(10.0, col('z')).alias('log10_z'),
).show()

# Salary formatting
print('=== Salary rounding ===')
people.filter(col('salary').isNotNull()).select(
    col('name'),
    col('salary'),
    round(col('salary'), 2).alias('salary_2dp'),
    round(col('salary'), -3).alias('rounded_to_1000'),
    ceil(col('salary') / 1000).alias('salary_k_ceil'),
).show(truncate=False)

## 3. Date and Time Functions

| Function | Description |
|----------|-------------|
| `current_date()` | Today's date as DateType (no args) |
| `current_timestamp()` | Current timestamp as TimestampType (no args) |
| `to_date(col, format)` | Parse string → DateType |
| `to_timestamp(col, format)` | Parse string → TimestampType |
| `date_format(col, fmt)` | Format Date/Timestamp → string |
| `date_add(col, n)` | Add n days |
| `date_sub(col, n)` | Subtract n days |
| `datediff(end, start)` | Days between two dates (end - start) |
| `months_between(end, start)` | Fractional months between two dates |
| `year(col)` | Extract year as IntegerType |
| `month(col)` | Extract month (1-12) |
| `dayofmonth(col)` | Extract day of month (1-31) |
| `dayofweek(col)` | Day of week (1=Sunday … 7=Saturday) |
| `hour(col)` | Extract hour (0-23) |
| `minute(col)` | Extract minute (0-59) |
| `second(col)` | Extract second (0-59) |
| `unix_timestamp(col, fmt)` | String → Unix epoch (Long) |
| `from_unixtime(col, fmt)` | Unix epoch → formatted string |

**Common date format strings:**
- `yyyy-MM-dd` → `2024-01-15`
- `MM/dd/yyyy` → `01/15/2024`
- `yyyy-MM-dd HH:mm:ss` → `2024-01-15 08:30:00`
- `MMM d, yyyy` → `Jan 15, 2024`

In [ ]:
# Cell 4: Date and time functions demo
from pyspark.sql.functions import (
    current_date, current_timestamp, to_date, to_timestamp,
    date_format, date_add, date_sub, datediff, months_between,
    year, month, dayofmonth, dayofweek, hour, minute, second,
    unix_timestamp, from_unixtime
)

# Parse strings to Date/Timestamp
print('=== to_date / to_timestamp ===')
events.select(
    col('event_date_str'),
    to_date(col('event_date_str'), 'yyyy-MM-dd').alias('event_date'),
    to_timestamp(col('event_ts_str'), 'yyyy-MM-dd HH:mm:ss').alias('event_ts'),
    from_unixtime(col('epoch'), 'yyyy-MM-dd HH:mm:ss').alias('from_epoch')
).show(truncate=False)

# Extract components
print('=== Date component extraction ===')
events_dt = events.withColumn('dt', to_date(col('event_date_str'), 'yyyy-MM-dd')) \
                  .withColumn('ts', to_timestamp(col('event_ts_str'), 'yyyy-MM-dd HH:mm:ss'))

events_dt.select(
    col('dt'),
    year(col('dt')).alias('yr'),
    month(col('dt')).alias('mo'),
    dayofmonth(col('dt')).alias('dom'),
    dayofweek(col('dt')).alias('dow'),   # 1=Sun, 2=Mon, ..., 7=Sat
    date_format(col('dt'), 'MMM d, yyyy').alias('formatted'),
    col('ts'),
    hour(col('ts')).alias('hr'),
    minute(col('ts')).alias('min'),
    second(col('ts')).alias('sec'),
).show(truncate=False)

# Date arithmetic
print('=== date_add / date_sub / datediff / months_between ===')
events_dt.select(
    col('dt'),
    date_add(col('dt'), 30).alias('plus_30d'),
    date_sub(col('dt'), 7).alias('minus_7d'),
    datediff(current_date(), col('dt')).alias('days_ago'),
    months_between(current_date(), col('dt')).alias('months_ago'),
).show(truncate=False)

# unix_timestamp
print('=== unix_timestamp ===')
events_dt.select(
    col('event_date_str'),
    unix_timestamp(col('event_date_str'), 'yyyy-MM-dd').alias('epoch_from_str'),
    col('epoch'),
).show()

## 4. Conditional / Null Functions

| Function | Signature | Description |
|----------|-----------|-------------|
| `when` | `when(cond, val).when(...).otherwise(default)` | CASE WHEN in Python |
| `coalesce` | `coalesce(col1, col2, ...)` | Returns first non-NULL value |
| `nullif` | `nullif(col1, col2)` | Returns NULL if col1 == col2, else col1 |
| `isnull` | `isnull(col)` | True if value is NULL |
| `isnotnull` | `isnotnull(col)` | True if value is not NULL |
| `ifnull` / `nvl` | SQL only: `IFNULL(col, default)` | Returns default if NULL (SQL string only) |
| `nvl2` | SQL only: `NVL2(col, not_null_val, null_val)` | Returns different values for NULL/not-NULL |

> **Exam note:** `isnull()` and `isnotnull()` are **functions** from `pyspark.sql.functions`.
> The **column methods** `.isNull()` and `.isNotNull()` (capital N) are called on a Column object.
> Both work — know both syntaxes.
>
> `coalesce(col1, col2, col3)` → SQL: `COALESCE(col1, col2, col3)` — evaluates left to right,
> returns first non-NULL. Use for fallback/default values.

In [ ]:
# Cell 5: Conditional / null functions demo
from pyspark.sql.functions import when, coalesce, nullif, isnull, isnotnull, lit

print('=== when / otherwise ===')
people.select(
    col('name'),
    col('age'),
    col('salary'),
    when(col('salary') >= 100000, 'Senior')
    .when(col('salary') >= 80000,  'Mid')
    .when(col('salary').isNotNull(), 'Junior')
    .otherwise('Unknown').alias('level'),
    when(col('age').isNull(), 'Age missing')
    .otherwise(col('age').cast('string')).alias('age_safe')
).show(truncate=False)

print('=== isnull / isnotnull / column .isNull() / .isNotNull() ===')
people.select(
    col('name'),
    col('email'),
    isnull(col('email')).alias('isnull_fn'),          # function approach
    col('email').isNull().alias('isNull_method'),      # column method approach
    isnotnull(col('age')).alias('age_present'),
).show()

print('=== coalesce — first non-null ===')
people.select(
    col('name'),
    col('email'),
    col('age'),
    coalesce(col('email'), lit('no-email@default.com')).alias('email_safe'),
    coalesce(col('age').cast('string'), lit('N/A')).alias('age_safe'),
).show(truncate=False)

print('=== nullif — returns NULL if values are equal ===')
spark.createDataFrame([
    (1, 'active',  'active'),
    (2, 'pending', 'active'),
    (3, 'closed',  'closed'),
], ['id', 'status', 'default_status']).select(
    col('id'),
    col('status'),
    col('default_status'),
    nullif(col('status'), col('default_status')).alias('status_if_changed')  # NULL when equal
).show()

print('=== ifnull / nvl2 via SQL ===')
people.selectExpr(
    'name',
    'email',
    'IFNULL(email, "unknown@example.com") AS email_ifnull',
    'NVL(email, "unknown@example.com") AS email_nvl',
    'NVL2(email, "Has email", "No email") AS has_email'
).show(truncate=False)

## 5. Aggregate Functions

| Function | Description | NULL behaviour |
|----------|-------------|----------------|
| `count('*')` | Count all rows including NULLs | Counts NULLs |
| `count(col)` | Count non-NULL values in column | Ignores NULLs |
| `countDistinct(col)` | Count distinct non-NULL values | Ignores NULLs |
| `sum(col)` | Sum of all non-NULL values | Ignores NULLs |
| `avg(col)` / `mean(col)` | Average of non-NULL values | Ignores NULLs |
| `min(col)` | Minimum value | Ignores NULLs |
| `max(col)` | Maximum value | Ignores NULLs |
| `collect_list(col)` | All values (including duplicates) as Array | Ignores NULLs |
| `collect_set(col)` | Unique values only as Array | Ignores NULLs |
| `first(col)` | First value in group | Can return NULL unless `ignorenulls=True` |
| `last(col)` | Last value in group | Can return NULL unless `ignorenulls=True` |
| `stddev(col)` | Sample standard deviation | Ignores NULLs |
| `variance(col)` | Sample variance | Ignores NULLs |

> **Exam gotcha:** `count('*')` counts ALL rows; `count(col('email'))` counts only non-NULL rows.
> `collect_list` preserves order and duplicates; `collect_set` returns unique values (no order guarantee).

In [ ]:
# Cell 6: Aggregate functions demo
from pyspark.sql.functions import (
    count, countDistinct, sum as F_sum, avg, mean, min as F_min, max as F_max,
    collect_list, collect_set, first, last, stddev, variance
)

sales = spark.createDataFrame([
    ('Alice',   'Engineering', 'Q1', 12000),
    ('Bob',     'Marketing',   'Q1', 9500),
    ('Carol',   'Engineering', 'Q2', 14000),
    ('Alice',   'Engineering', 'Q2', 13500),
    ('Bob',     'Marketing',   'Q2', 9500),    # duplicate amount
    ('Dave',    'HR',          'Q1', None),    # null value
    ('Eve',     'Engineering', 'Q1', 11000),
], ['rep', 'dept', 'quarter', 'revenue'])

# NULL behaviour of count
print('=== count(*) vs count(col) — NULL handling ===')
sales.agg(
    count('*').alias('all_rows'),                   # includes NULL row
    count(col('revenue')).alias('non_null_rev'),    # excludes NULL
    countDistinct(col('dept')).alias('unique_depts'),
    countDistinct(col('quarter')).alias('unique_quarters'),
).show()

# Aggregations per group
print('=== groupBy with all agg functions ===')
sales.groupBy('dept').agg(
    count('*').alias('rows'),
    count(col('revenue')).alias('non_null_revenues'),
    F_sum('revenue').alias('total_rev'),
    avg('revenue').alias('avg_rev'),
    F_min('revenue').alias('min_rev'),
    F_max('revenue').alias('max_rev'),
    stddev('revenue').alias('stddev_rev'),
    collect_list('revenue').alias('all_revenues'),     # includes duplicates
    collect_set('revenue').alias('unique_revenues'),   # unique values
    first('rep').alias('first_rep'),
    last('rep').alias('last_rep'),
).show(truncate=False)

# first/last with ignorenulls
print('=== first / last with ignorenulls ===')
sales.agg(
    first('revenue').alias('first_rev'),
    first('revenue', ignorenulls=True).alias('first_non_null_rev'),
    last('revenue').alias('last_rev'),
    last('revenue', ignorenulls=True).alias('last_non_null_rev'),
).show()

## 6. Array / Collection Functions

| Function | Description |
|----------|-------------|
| `array(col1, col2, ...)` | Create an array column from multiple columns |
| `explode(col)` | One row per array element; skips NULLs and empty arrays |
| `explode_outer(col)` | Like `explode` but keeps rows with NULL/empty (emits NULL) |
| `posexplode(col)` | Like `explode` but also returns position index |
| `array_contains(col, value)` | True if array contains value |
| `size(col)` | Length of array or map; returns -1 for NULL |
| `flatten(col)` | Flatten ArrayType(ArrayType) → single-level array |
| `sort_array(col)` | Sort array elements ascending (default) |
| `array_distinct(col)` | Remove duplicates from array |
| `array_union(col1, col2)` | Union of two arrays (distinct elements) |
| `map_keys(col)` | Extract keys from MapType column as array |
| `map_values(col)` | Extract values from MapType column as array |

> **`explode` vs `explode_outer`:**
> - `explode([])` or `explode(NULL)` → **row is dropped**
> - `explode_outer([])` or `explode_outer(NULL)` → **NULL row is emitted**

In [ ]:
# Cell 7: Array / collection functions demo
from pyspark.sql.functions import (
    array, explode, explode_outer, posexplode,
    array_contains, size, flatten, sort_array,
    array_distinct, array_union, map_keys, map_values
)

print('=== items DataFrame ===')
items.show(truncate=False)

# size / array_contains / sort_array / array_distinct
print('=== size / array_contains / sort_array / array_distinct ===')
items.select(
    col('id'),
    col('fruits'),
    size(col('fruits')).alias('num_fruits'),
    array_contains(col('fruits'), 'apple').alias('has_apple'),
    sort_array(col('fruits')).alias('sorted_fruits'),
    array_distinct(col('fruits')).alias('distinct_fruits'),
).show(truncate=False)

# explode vs explode_outer
print('=== explode — drops empty arrays ===')
items.select(col('id'), explode(col('fruits')).alias('fruit')).show()

print('=== explode_outer — keeps rows with empty/null arrays ===')
items.select(col('id'), explode_outer(col('fruits')).alias('fruit')).show()

# posexplode — returns (pos, element)
print('=== posexplode — includes position index ===')
items.filter(size(col('fruits')) > 0) \
     .select(col('id'), posexplode(col('fruits')).alias('pos', 'fruit')).show()

# map_keys / map_values
print('=== map_keys / map_values ===')
items.select(
    col('id'),
    col('scores'),
    map_keys(col('scores')).alias('score_keys'),
    map_values(col('scores')).alias('score_values'),
).show(truncate=False)

# array / flatten / array_union
print('=== array / array_union / flatten ===')
pairs = spark.createDataFrame([
    (1, ['a', 'b'], ['b', 'c', 'd']),
    (2, ['x'],      ['y', 'z']),
], ['id', 'arr1', 'arr2'])

pairs.select(
    col('id'),
    array(col('id'), F.lit(99)).alias('created_array'),
    array_union(col('arr1'), col('arr2')).alias('union_arr'),
    flatten(array(col('arr1'), col('arr2'))).alias('flat_arr'),
).show(truncate=False)

In [ ]:
# Cell 8: SQL string equivalents — all functions work inside spark.sql() / selectExpr()
people.createOrReplaceTempView('people')
events_dt.createOrReplaceTempView('events_dt')
items.createOrReplaceTempView('items')

print('=== SQL equivalents — same functions, no import needed ===')
spark.sql("""
    SELECT
        INITCAP(TRIM(name))                       AS clean_name,
        COALESCE(email, 'unknown@default.com')    AS email_safe,
        ROUND(salary, 0)                          AS salary_rounded,
        CASE
            WHEN salary >= 100000 THEN 'Senior'
            WHEN salary >= 80000  THEN 'Mid'
            WHEN salary IS NOT NULL THEN 'Junior'
            ELSE 'Unknown'
        END                                       AS level,
        NVL2(email, 'Has email', 'No email')      AS email_status,
        NULLIF(age, 0)                            AS age_nullif_zero
    FROM people
""").show(truncate=False)

# selectExpr — inline SQL on a DataFrame without registering a view
print('=== selectExpr equivalents ===')
people.selectExpr(
    'UPPER(TRIM(name)) AS upper_name',
    'REGEXP_REPLACE(email, "@.*", "") AS username',
    'ROUND(salary, -3) AS salary_k',
    'DATEDIFF(CURRENT_DATE(), CAST(2019 AS STRING)) AS fake_days',  # illustrative
    'CASE WHEN age IS NULL THEN -1 ELSE age END AS age_safe',
).show(truncate=False)

spark.stop()
print('Done.')

## 7. Real-World Scenarios

<details>
<summary>Scenario 1: Cleansing Customer Names and Emails with String Functions</summary>

**Situation:**
A CRM system exports customer data with inconsistent casing, leading/trailing whitespace,
and special characters in names. Before loading into a data warehouse, the team must
standardise all string fields.

**Code:**
```python
from pyspark.sql.functions import trim, initcap, lower, regexp_replace, regexp_extract

cleaned = raw_customers.select(
    col('customer_id'),
    initcap(trim(col('first_name'))).alias('first_name'),
    initcap(trim(col('last_name'))).alias('last_name'),
    lower(trim(col('email'))).alias('email'),
    regexp_replace(col('phone'), r'[^0-9]', '').alias('phone_digits'),
    regexp_extract(col('email'), r'@([^.]+)\.', 1).alias('email_domain'),
)
cleaned.show()
```

**Expected Outcome:**
Names are properly capitalised, emails are lowercased, phone numbers contain only digits,
and the email domain is extracted for segmentation.

**Exam Sub-topic:** `trim`, `initcap`, `lower`, `regexp_replace`, `regexp_extract`
</details>

<details>
<summary>Scenario 2: Date-Based Cohort Analysis with Date Functions</summary>

**Situation:**
A subscription analytics team needs to compute cohort metrics: which month each user
signed up, how many months they've been active, and whether their subscription is
within the renewal window (30 days before expiry).

**Code:**
```python
from pyspark.sql.functions import (
    to_date, year, month, datediff, months_between,
    date_add, current_date, date_format
)

subscriptions = spark.read.parquet('/data/subscriptions/')

enriched = subscriptions.withColumn(
    'signup_date', to_date(col('signup_date_str'), 'yyyy-MM-dd')
).withColumn(
    'expiry_date', to_date(col('expiry_date_str'), 'yyyy-MM-dd')
).withColumn(
    'cohort', date_format(col('signup_date'), 'yyyy-MM')  # e.g., '2024-01'
).withColumn(
    'months_active', months_between(current_date(), col('signup_date')).cast('int')
).withColumn(
    'days_to_expiry', datediff(col('expiry_date'), current_date())
).withColumn(
    'in_renewal_window',
    (col('days_to_expiry') >= 0) & (col('days_to_expiry') <= 30)
)

# Cohort summary
enriched.groupBy('cohort').agg(
    count('*').alias('subscribers'),
    avg('months_active').alias('avg_months'),
    sum(col('in_renewal_window').cast('int')).alias('up_for_renewal')
).orderBy('cohort').show()
```

**Expected Outcome:**
Cohort table showing subscriber counts, average tenure, and renewal pipeline per cohort month.

**Exam Sub-topic:** `to_date`, `date_format`, `months_between`, `datediff`, `current_date`
</details>

<details>
<summary>Scenario 3: Expanding Event Tags Array into Individual Rows for Analytics</summary>

**Situation:**
A web analytics platform stores page view events with an array of tags per event.
The analytics team needs a flat table with one row per (event, tag) to count
tag frequencies. Some events have no tags and must be preserved.

**Code:**
```python
from pyspark.sql.functions import explode_outer, array_distinct, size

events = spark.read.parquet('/data/page_events/')
# Schema: event_id, page, tags: ArrayType(StringType), timestamp

# Flatten: one row per tag, keep events with no tags as NULL row
flat_tags = events.select(
    col('event_id'),
    col('page'),
    col('timestamp'),
    explode_outer(col('tags')).alias('tag'),  # keeps zero-tag events
)

# Tag frequency analysis
tag_counts = flat_tags.filter(col('tag').isNotNull()) \
    .groupBy('tag') \
    .agg(count('*').alias('event_count')) \
    .orderBy(col('event_count').desc())

tag_counts.show(20)

# Report events with no tags
no_tag_events = flat_tags.filter(col('tag').isNull()) \
    .select('event_id', 'page', 'timestamp')
print(f'Events with no tags: {no_tag_events.count()}')
```

**Expected Outcome:**
Top tags by frequency. Events with no tags are preserved (thanks to `explode_outer`).
Using `explode` instead would silently drop those rows.

**Exam Sub-topic:** `explode` vs `explode_outer`; `array_contains`; `size`
</details>

<details>
<summary>Scenario 4: Revenue Band Classification Using when() and Numeric Functions</summary>

**Situation:**
A finance team needs to classify transactions into revenue bands, compute rounded
summaries, and flag outliers (transactions more than 2 standard deviations from mean).

**Code:**
```python
from pyspark.sql.functions import when, round, abs as F_abs, stddev, avg, sqrt

# Compute global stats
stats = transactions.agg(
    avg('amount').alias('mean_amt'),
    stddev('amount').alias('std_amt')
).collect()[0]

mean_val = stats['mean_amt']
std_val  = stats['std_amt']
threshold = 2 * std_val

enriched = transactions.withColumn(
    'amount_rounded', round(col('amount'), 2)
).withColumn(
    'revenue_band',
    when(col('amount') >= 10000, 'Large')
    .when(col('amount') >= 1000,  'Medium')
    .when(col('amount') >= 100,   'Small')
    .otherwise('Micro')
).withColumn(
    'is_outlier',
    F_abs(col('amount') - mean_val) > threshold
)

enriched.groupBy('revenue_band').agg(
    count('*').alias('count'),
    round(F_sum('amount'), 2).alias('total'),
    sum(col('is_outlier').cast('int')).alias('outliers')
).show()
```

**Expected Outcome:**
Revenue band summary with outlier counts flagged for audit review.

**Exam Sub-topic:** `when`/`otherwise`; `round`; `abs`; `stddev`; aggregate functions
</details>

<details>
<summary>Scenario 5: Null-Safe Data Merging with coalesce() and collect_set()</summary>

**Situation:**
A data quality pipeline merges records from two sources (primary and fallback).
If the primary source has a NULL for a field, use the fallback value.
After merging, collect all unique source tags per customer into an array.

**Code:**
```python
from pyspark.sql.functions import coalesce, collect_set, concat_ws, size

# Merge primary + fallback with coalesce
merged = primary.join(fallback, on='customer_id', how='left').select(
    col('customer_id'),
    coalesce(primary['name'],    fallback['name']).alias('name'),
    coalesce(primary['email'],   fallback['email']).alias('email'),
    coalesce(primary['phone'],   fallback['phone']).alias('phone'),
    coalesce(primary['country'], fallback['country'], F.lit('UNKNOWN')).alias('country'),
)

# Aggregate: unique data sources per customer
source_summary = events.groupBy('customer_id').agg(
    collect_set('source_tag').alias('sources'),           # unique tags only
    size(collect_set('source_tag')).alias('source_count'),
    concat_ws(', ', collect_set('source_tag')).alias('sources_csv'),
)

final = merged.join(source_summary, on='customer_id', how='left')
final.show(truncate=False)
```

**Expected Outcome:**
All customers have non-NULL core fields (fell back to secondary source where needed).
`collect_set` ensures no duplicate source tags appear in the summary array.

**Exam Sub-topic:** `coalesce` for null fallback; `collect_set` vs `collect_list`; `size`; `concat_ws`
</details>

## 8. Exam Cheat Sheet

| Category | Key Exam Facts |
|----------|---------------|
| Import | `from pyspark.sql.functions import ...` — required for DataFrame API |
| SQL strings | No import needed inside `spark.sql()`, `selectExpr()`, `expr()` |
| `count('*')` | Counts ALL rows including NULLs |
| `count(col)` | Counts only non-NULL values |
| `collect_list` | Keeps duplicates; order not guaranteed |
| `collect_set` | Unique values only; order not guaranteed |
| `explode` | Drops rows with NULL or empty arrays |
| `explode_outer` | Keeps rows with NULL/empty (emits NULL element) |
| `coalesce(c1, c2)` | First non-NULL — evaluates left to right |
| `nullif(c1, c2)` | Returns NULL if c1 == c2, else c1 |
| `round(x, 0)` | HALF_UP: 2.5 → 3.0 |
| `bround(x, 0)` | HALF_EVEN (banker's): 2.5 → 2.0 |
| `datediff(end, start)` | end − start in days (can be negative) |
| `isnull(col)` | Function from `pyspark.sql.functions` |
| `col.isNull()` | Column method (capital N) — same result |
| `split(col, pat)` | Returns ArrayType |
| `concat(c1, c2)` | NULL in any arg → result is NULL |
| `concat_ws(sep, ...)` | NULLs are skipped (not propagated) |
| `size(arr)` | Returns -1 for NULL array |
| `dayofweek(dt)` | 1=Sunday, 2=Monday, ..., 7=Saturday |

---

## 9. Practice Q&A

<details>
<summary>Q1: What is the difference between concat() and concat_ws()?</summary>

**A:**
- `concat(col1, col2, col3)` — concatenates values. If **any** argument is NULL, the entire result is NULL.
- `concat_ws(sep, col1, col2, col3)` — concatenates with a separator. **NULLs are skipped** (ignored), not propagated.

```python
df = spark.createDataFrame([('Alice', None, 'Smith')], ['first', 'middle', 'last'])
df.select(
    concat(col('first'), lit(' '), col('middle'), lit(' '), col('last')).alias('concat'),
    concat_ws(' ', col('first'), col('middle'), col('last')).alias('concat_ws'),
).show()
# concat  → null         (NULL propagated)
# concat_ws → 'Alice Smith'  (NULL skipped)
```
</details>

<details>
<summary>Q2: What is the difference between explode() and explode_outer()?</summary>

**A:**
- `explode(array_col)` — expands each array element into a separate row. Rows with **NULL or empty arrays are dropped**.
- `explode_outer(array_col)` — same expansion, but rows with NULL or empty arrays are **kept as a single row with NULL** in the exploded column.

Use `explode_outer` when you need to preserve parent rows even if they have no array elements (e.g., for LEFT JOIN-style semantics).
</details>

<details>
<summary>Q3: How does count() handle NULL values?</summary>

**A:**
- `count('*')` — counts **all rows**, including rows where the column is NULL
- `count(col('column_name'))` — counts only **non-NULL values** in that column
- `countDistinct(col('column_name'))` — counts **distinct non-NULL values**

This is a common exam trap: given a column with 10 rows where 3 are NULL:
- `count('*')` → 10
- `count('column_name')` → 7
- `countDistinct('column_name')` → number of unique non-NULL values
</details>

<details>
<summary>Q4: What is the difference between collect_list() and collect_set()?</summary>

**A:**
- `collect_list(col)` — aggregates all values into an array, **preserving duplicates**. Order is not guaranteed.
- `collect_set(col)` — aggregates into an array of **unique values only** (deduplication). Order is not guaranteed.

Both ignore NULLs. Use `collect_set` when you need distinct values; use `collect_list` when you want all occurrences (e.g., audit trail, event sequences).
</details>

<details>
<summary>Q5: When should you use coalesce() vs when().otherwise()?</summary>

**A:**
- `coalesce(col1, col2, col3)` — use when you want the **first non-NULL value** from a list of columns (null fallback chain). Clean and concise for this pattern.
- `when(cond, val).when(...).otherwise(default)` — use when the **condition is not just null-checking** — e.g., when you need different transformations based on value ranges, types, or complex conditions.

```python
# coalesce: pick first non-null
coalesce(col('phone_mobile'), col('phone_home'), lit('UNKNOWN'))

# when/otherwise: different logic per value
when(col('score') >= 90, 'A').when(col('score') >= 80, 'B').otherwise('C')
```
</details>